# YOLOv8 Fine-Tuning for APCOMS Shuttle Passenger Detection

Fine-tunes the base YOLOv8n model on shuttle entrance footage so person
detection is tuned to our deployment camera angle, lighting conditions,
and demographic.

**Dataset:** ~300 labeled frames from simulated Makerere shuttle entrance footage with diverse people.

**Output:** A `best.pt` file that replaces `counting_module/models/yolov8n.pt`
for improved detection accuracy in production.

---

## 0. Environment setup

In [ ]:
# Mount Google Drive so we can save weights persistently across sessions.
# When prompted, sign in with the SAME Gmail used to create APCOMS_YOLO_runs.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install ultralytics (provides the YOLO class and CLI). Pinned to a
# stable version so future runs reproduce today's training behaviour.
!pip install -q ultralytics==8.3.0

In [ ]:
# Confirm GPU availability. Should print T4, A100, or similar.
# If this prints "command not found" or no GPU, go to
# Runtime > Change runtime type > Hardware accelerator > T4 GPU.
!nvidia-smi

---

## 1. Dataset download

The labeled dataset lives in Roboflow (set up in Chunk 4 of the
fine-tuning workflow). Roboflow exports a YOLO-format zip with the
correct folder structure: images and labels split into train/val/test.

Replace the placeholder URL below with the actual export link once
labeling is complete.

In [ ]:
# Working directory for this session. Colab's local filesystem gets
# wiped when the session ends, but that's fine -- we save final
# artifacts to Drive at the end. /content is just our scratch space.
import os
os.chdir('/content')

# Roboflow YOLO-format dataset URL.
# PLACEHOLDER -- replace with the actual export URL once Chunk 4
# is complete. Roboflow gives you a download URL ending in
# .../export/yolov8 or similar.
DATASET_URL = "PASTE_ROBOFLOW_EXPORT_URL_HERE"

# Fetch and unzip the dataset. The unzipped folder will contain:
#   dataset/
#     data.yaml          (class list + train/val/test paths)
#     train/images/*.jpg
#     train/labels/*.txt
#     valid/images/*.jpg
#     valid/labels/*.txt
#     test/images/*.jpg
#     test/labels/*.txt
!curl -L "$DATASET_URL" -o dataset.zip
!unzip -q dataset.zip -d dataset
!ls dataset

In [ ]:
# Verify dataset structure. data.yaml is the file YOLO reads to know
# class names and folder locations. Print it so we can spot issues
# (wrong paths, wrong class count, etc.) before training kicks off.
!cat dataset/data.yaml

---

## 2. Training configuration

Settings tuned for a small, on-distribution dataset (~300-500 frames).
Key choices and why:

- **epochs=100**: enough for the model to converge on a small dataset
  without overfitting catastrophically. We'll monitor val loss and can
  stop early if it diverges.
- **imgsz=640**: standard YOLOv8 input size. Bigger means slower training
  and inference but better small-object detection. 640 is the sweet spot
  for our use case.
- **batch=16**: fits comfortably in T4 GPU memory. Lower if you hit OOM.
- **patience=20**: early stopping if val loss hasn't improved in 20
  epochs. Saves time when the model has already converged.
- **augment=True**: YOLOv8's built-in augmentation -- mosaic, mixup,
  HSV shifts, flips. Compensates for low person-diversity in our dataset.

In [ ]:
# Training hyperparameters. All in one place so future tweaks are easy.
TRAINING_CONFIG = {
    "data": "/content/dataset/data.yaml",
    "epochs": 100,
    "imgsz": 640,
    "batch": 16,
    "patience": 20,
    "project": "/content/drive/MyDrive/APCOMS_YOLO_runs",
    "name": "shuttle_finetune_v1",
    "exist_ok": True,
    "augment": True,
    "verbose": True,
}

# Sanity check: print the config so we can confirm before kicking off
# a training run that might take 15-30 minutes.
for key, value in TRAINING_CONFIG.items():
    print(f"  {key}: {value}")

---

## 3. Train the model

Loads base YOLOv8n weights (pre-trained on COCO) and continues
training on our shuttle dataset. This is "transfer learning" -- we
keep COCO's general feature extractors and just adjust the final
layers to specialize for shuttle entrance detection.

Expected runtime: 15-30 minutes on a T4 GPU. Live metrics print
after each epoch. Best weights are saved automatically to
APCOMS_YOLO_runs/shuttle_finetune_v1/weights/best.pt

In [ ]:
from ultralytics import YOLO

# Load base COCO weights as our starting point. Ultralytics downloads
# the file automatically the first time it's referenced.
model = YOLO("yolov8n.pt")

# Kick off training. The unpacked TRAINING_CONFIG dict provides every
# argument YOLO needs -- keeps the call signature clean.
results = model.train(**TRAINING_CONFIG)

---

## 4. Validate the fine-tuned model

Evaluates our trained model on the held-out test set -- frames the
model never saw during training. This gives us an honest accuracy
number.

Metrics produced:
- **mAP@50**: mean Average Precision at IoU threshold 0.5.
  YOLOv8's headline accuracy number -- higher is better.
- **mAP@50-95**: stricter version (averaged across IoU thresholds
  0.5 to 0.95). The standard YOLO benchmark metric.
- **Precision**: of all detections, what fraction were correct.
- **Recall**: of all real people, what fraction did we detect.

For shuttle counting, recall matters slightly more than precision --
missing a passenger is worse than briefly seeing a phantom one
(which the tracker filters out anyway).

In [ ]:
# Load the freshly-trained best.pt from this run.
best_weights_path = (
    "/content/drive/MyDrive/APCOMS_YOLO_runs/"
    "shuttle_finetune_v1/weights/best.pt"
)
finetuned_model = YOLO(best_weights_path)

# Run validation on the test split. data.yaml already points YOLO
# at the correct test images and labels.
val_results = finetuned_model.val(
    data="/content/dataset/data.yaml",
    split="test",
    imgsz=640,
    verbose=True,
)

print("\nFine-tuned model metrics on test set:")
print(f"  mAP@50:     {val_results.box.map50:.4f}")
print(f"  mAP@50-95:  {val_results.box.map:.4f}")
print(f"  Precision:  {val_results.box.mp:.4f}")
print(f"  Recall:     {val_results.box.mr:.4f}")

---

## 5. Compare against the base COCO model

The whole point of fine-tuning is producing a model BETTER than the
generic base. Here we run the base YOLOv8n (untouched, COCO-trained)
on the exact same test set and compare metrics.

This side-by-side comparison is the headline result for the project
report and panel presentation. Even modest improvements
(say, +5% recall) are meaningful evidence that domain-specific
fine-tuning pays off.

In [ ]:
# Load the untouched base model -- exactly what counting_module ships
# with today. Ultralytics caches the download from Section 3 so this
# is instant.
base_model = YOLO("yolov8n.pt")

# Identical validation call, just on the base model. Same test images,
# same image size, same metric calculation -- only the weights differ.
base_results = base_model.val(
    data="/content/dataset/data.yaml",
    split="test",
    imgsz=640,
    verbose=True,
)

print("\nBase COCO model metrics on test set:")
print(f"  mAP@50:     {base_results.box.map50:.4f}")
print(f"  mAP@50-95:  {base_results.box.map:.4f}")
print(f"  Precision:  {base_results.box.mp:.4f}")
print(f"  Recall:     {base_results.box.mr:.4f}")

In [ ]:
# Side-by-side comparison table. Prints the absolute numbers AND
# the deltas so we can see at a glance how much fine-tuning helped.
def pct_delta(new, old):
    if old == 0:
        return "n/a"
    return f"{(new - old) * 100:+.2f}pp"

print(f"\n{'Metric':<14}{'Base COCO':>12}{'Fine-tuned':>14}{'Delta':>12}")
print("-" * 52)
print(
    f"{'mAP@50':<14}"
    f"{base_results.box.map50:>12.4f}"
    f"{val_results.box.map50:>14.4f}"
    f"{pct_delta(val_results.box.map50, base_results.box.map50):>12}"
)
print(
    f"{'mAP@50-95':<14}"
    f"{base_results.box.map:>12.4f}"
    f"{val_results.box.map:>14.4f}"
    f"{pct_delta(val_results.box.map, base_results.box.map):>12}"
)
print(
    f"{'Precision':<14}"
    f"{base_results.box.mp:>12.4f}"
    f"{val_results.box.mp:>14.4f}"
    f"{pct_delta(val_results.box.mp, base_results.box.mp):>12}"
)
print(
    f"{'Recall':<14}"
    f"{base_results.box.mr:>12.4f}"
    f"{val_results.box.mr:>14.4f}"
    f"{pct_delta(val_results.box.mr, base_results.box.mr):>12}"
)

---

## 6. Export final weights

The best-performing weights from training live in Drive at:
`APCOMS_YOLO_runs/shuttle_finetune_v1/weights/best.pt`

To deploy: download that file and replace `counting_module/models/yolov8n.pt`
in the APCOMS repo. The counting pipeline will automatically pick up
the new weights on its next run -- no code changes needed because
ObjectDetection reads from the same path.

For panel demos, keep BOTH versions: the original COCO weights and
the fine-tuned weights, so you can show before/after detection
quality if questioned.

In [ ]:
# Confirm the trained weights actually landed in Drive and print
# the size as a sanity check. Expect ~6 MB for YOLOv8n.
import os

weights_path = (
    "/content/drive/MyDrive/APCOMS_YOLO_runs/"
    "shuttle_finetune_v1/weights/best.pt"
)

if os.path.exists(weights_path):
    size_mb = os.path.getsize(weights_path) / (1024 * 1024)
    print(f"Fine-tuned weights ready at:")
    print(f"  {weights_path}")
    print(f"  Size: {size_mb:.2f} MB")
    print("\nDownload this file from Drive and place it at:")
    print("  counting_module/models/yolov8n.pt")
else:
    print(f"WARNING: weights not found at {weights_path}")
    print("Check that Section 3 (training) completed successfully.")

In [ ]:
# Optional: list everything saved in the training run folder. Useful
# for finding the training plots (results.png, confusion_matrix.png,
# PR curves) that we can include in the project report and panel
# slides as visual evidence of what training looked like.
run_dir = "/content/drive/MyDrive/APCOMS_YOLO_runs/shuttle_finetune_v1"
print(f"Contents of {run_dir}:")
for root, dirs, files in os.walk(run_dir):
    level = root.replace(run_dir, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/")
    sub_indent = '  ' * (level + 1)
    for file in files:
        print(f"{sub_indent}{file}")